### 1 — Load the Cleaned Dataset
We load the pre-processed F1 2024 lap data (with feature engineering) from the `clean` folder.
This dataset already includes features like `stint_mean`, `lap_time_rel`, and `lap_time_norm`.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/tyre_project')

import pandas as pd

df = pd.read_csv('clean/clean_laps_2024_fe.csv')
df.head()


Mounted at /content/drive


,driver,team,lap_number,LapTime,compound,is_pit,stint,lap_start_time,lap_time_s,year,event,round,stint_index,lap_index_in_stint,stint_length,stint_mean,lap_time_rel,lap_time_norm
0,ALB,Williams,1,0 days 00:01:43.888000,SOFT,False,1.0,0 days 00:59:59.911000,103.888,2024,Bahrain Grand Prix,1,1,0,56,98.511214,5.376786,1.054580
1,ALO,Aston Martin,1,0 days 00:01:41.679000,SOFT,False,1.0,0 days 00:59:59.911000,101.679,2024,Bahrain Grand Prix,1,1,0,57,97.888228,3.790772,1.038726
2,BOT,Kick Sauber,1,0 days 00:01:48.536000,SOFT,False,1.0,0 days 00:59:59.911000,108.536,2024,Bahrain Grand Prix,1,1,0,55,98.503745,10.032255,1.101846
3,GAS,Alpine,1,0 days 00:01:47.240000,SOFT,False,1.0,0 days 00:59:59.911000,107.240,2024,Bahrain Grand Prix,1,1,0,56,98.877839,8.362161,1.084571
4,HAM,Mercedes,1,0 days 00:01:43.122000,SOFT,False,1.0,0 days 00:59:59.911000,103.122,2024,Bahrain Grand Prix,1,1,0,57,97.457298,5.664702,1.058125


### 2 — Prepare Inputs for Modelling
We select the features and the target variable that will be used for predicting lap times.

• Features:
   - lap_index_in_stint
   - compound
   - driver
   - event

• Target:
   - lap_time_s

Categorical variables are one-hot encoded using OneHotEncoder,
while the numeric variable (lap_index_in_stint) is passed through without transformation.


In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error

features = ['lap_index_in_stint', 'compound', 'driver', 'event']
target = 'lap_time_s'

X = df[features]
y = df[target]

categorical = ['compound', 'driver', 'event']
numeric = ['lap_index_in_stint']

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('num', 'passthrough', numeric)
])


**## 3. Advanced Machine Learning Models (Report 3)**

In this section, we extend the baseline models used in Report 2 by introducing non-linear machine learning algorithms.  
These models are able to capture the true non-linear behaviour of tyre degradation, lap-time drop-off, and stint evolution — something simple linear models cannot represent accurately.

The goal of this section is to build stronger predictive models that better describe how tyre wear affects performance over long runs.


### **3.1 — Random Forest Regressor (Non-Linear Model)**

In this section, we introduce Random Forest, a tree-based ensemble method that captures non-linear relationships in tyre-degradation data.

Unlike Linear and Ridge Regression (which assume a straight-line relationship), Random Forest can model:

* tyre wear drop-off curves
* interactions between compound × driver × event
* sharp pace changes during stints
* non-linear patterns in lap evolution

This makes it a much stronger model for predicting F1 lap times.

The model is wrapped in a Pipeline so that:

* the same One-Hot Encoding used before is applied
* the model receives clean numerical features
* everything remains reproducible and consistent across models

We fit the model and compute the MAE to compare against the baseline models.


In [4]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape


((21067, 4), (5267, 4))

In [5]:
from sklearn.ensemble import RandomForestRegressor

rf = Pipeline([
    ('preprocess', preprocess),
    ('model', RandomForestRegressor(
        n_estimators=200,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
preds_rf = rf.predict(X_test)

print("Random Forest MAE:", mean_absolute_error(y_test, preds_rf))


Random Forest MAE: 1.5608391627112257


### **3.2 Gradient Boosting Regressor — Results & Interpretation**

The Gradient Boosting Regressor was trained to model the non-linear and compound-dependent behaviour of tyre degradation.  
This model is well-suited for F1 lap-time prediction because:

* Lap-time evolution over a stint is highly non-linear
* Tyre compounds affect degradation differently
* Drivers and tracks interact in complex ways
* Boosting sequentially corrects mistakes made by previous weak learners

**Model Performance:**

* MAE = 3.19 seconds  
  → This is a significant improvement over Linear and Ridge Regression (~3.71 s).  
  → This demonstrates that boosting models capture non-linear degradation patterns missed by simpler baselines.

The strong performance suggests that Gradient Boosting can model complex driver-track-tyre interactions and provides a much more accurate foundation for predicting stint evolution.


In [6]:
from sklearn.ensemble import GradientBoostingRegressor

gb = Pipeline([
    ('preprocess', preprocess),
    ('model', GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

gb.fit(X_train, y_train)
preds_gb = gb.predict(X_test)

print("Gradient Boosting MAE:", mean_absolute_error(y_test, preds_gb))


Gradient Boosting MAE: 3.1932779267735234


### **Feature Importance Analysis (Gradient Boosting)**

To understand which factors influence tyre degradation prediction the most, we analyze the feature importances learned by the Gradient Boosting model.

This tells us:

* which circuits contribute the most to prediction variance
* which drivers influence tyre degradation patterns
* whether stint progression (lap index) is more important than categorical factors

We extract the one-hot-encoded feature names and compute importances for all features.


In [7]:
import pandas as pd
import numpy as np

# Extract feature names after one-hot encoding
onehot = gb.named_steps['preprocess'].named_transformers_['cat']
feature_names_cat = onehot.get_feature_names_out(['compound', 'driver', 'event'])
feature_names = np.concatenate((feature_names_cat, ['lap_index_in_stint']))

# Extract importance values
importances = gb.named_steps['model'].feature_importances_

# Create DataFrame
fi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values(by='importance', ascending=False)

fi_df.head(20)


,feature,importance
53,lap_index_in_stint,0.200403
31,event_Austrian Grand Prix,0.154986
38,event_Dutch Grand Prix,0.095953
32,event_Azerbaijan Grand Prix,0.083079
34,event_Belgian Grand Prix,0.074553
37,event_Chinese Grand Prix,0.067751
46,event_Monaco Grand Prix,0.043183
50,event_Spanish Grand Prix,0.038604
39,event_Emilia Romagna Grand Prix,0.029945
44,event_Mexico City Grand Prix,0.029826


### **4 — Cross-Race Validation (Leave-One-Event-Out) for Advanced Models**

To evaluate how well the non-linear models generalize across different circuits, we apply **Leave-One-Event-Out validation**.
For each Grand Prix:

1. Train on all races *except* one
2. Test on the held-out race
3. Compute MAE
4. Repeat for all 24 races

This evaluates whether the model can generalize to unseen tracks with different degradation patterns.

We apply this method to:

* **Random Forest Regressor**
* **Gradient Boosting Regressor**


In [10]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor

events = sorted(df['event'].unique())
rf_event_results = []

for ev in events:
    # Split data
    train_df = df[df['event'] != ev]
    test_df = df[df['event'] == ev]

    X_train_ev = train_df[features]
    y_train_ev = train_df[target]
    X_test_ev = test_df[features]
    y_test_ev = test_df[target]

    # Pipeline
    rf_event_model = Pipeline([
        ('preprocess', preprocess),
        ('model', RandomForestRegressor(
            n_estimators=80,
            max_depth=12,
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Train & predict
    rf_event_model.fit(X_train_ev, y_train_ev)
    preds = rf_event_model.predict(X_test_ev)

    # Compute MAE
    mae = mean_absolute_error(y_test_ev, preds)
    rf_event_results.append((ev, mae))

rf_event_df = pd.DataFrame(rf_event_results, columns=['event', 'MAE'])
rf_event_df.sort_values(by='MAE')


,event,MAE
19,Saudi Arabian Grand Prix,2.857375
16,Miami Grand Prix,3.912644
4,Bahrain Grand Prix,4.934679
13,Japanese Grand Prix,5.146311
6,British Grand Prix,5.238039
14,Las Vegas Grand Prix,5.598078
0,Abu Dhabi Grand Prix,5.952772
20,Singapore Grand Prix,7.487172
23,United States Grand Prix,8.206253
12,Italian Grand Prix,10.146199


In [11]:
from sklearn.ensemble import GradientBoostingRegressor

gb_event_results = []

for ev in events:
    train_df = df[df['event'] != ev]
    test_df = df[df['event'] == ev]

    X_train_ev = train_df[features]
    y_train_ev = train_df[target]
    X_test_ev = test_df[features]
    y_test_ev = test_df[target]

    gb_event_model = Pipeline([
        ('preprocess', preprocess),
        ('model', GradientBoostingRegressor(
            n_estimators=50,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ])

    gb_event_model.fit(X_train_ev, y_train_ev)
    preds = gb_event_model.predict(X_test_ev)

    mae = mean_absolute_error(y_test_ev, preds)
    gb_event_results.append((ev, mae))

gb_event_df = pd.DataFrame(gb_event_results, columns=['event', 'MAE'])
gb_event_df.sort_values(by='MAE')


,event,MAE
0,Abu Dhabi Grand Prix,3.337471
19,Saudi Arabian Grand Prix,3.896292
16,Miami Grand Prix,4.484938
6,British Grand Prix,5.451505
4,Bahrain Grand Prix,6.831006
11,Hungarian Grand Prix,7.253256
12,Italian Grand Prix,7.719372
14,Las Vegas Grand Prix,8.168662
13,Japanese Grand Prix,8.235316
20,Singapore Grand Prix,8.411675


### **4.1 Cross-Race Validation Results — Interpretation**

The results show clear differences in how each model generalizes across circuits.

#### **Random Forest**

* Performs well on stable, low-degradation circuits
* Struggles on high-variance tracks like Belgium, Austria, and Azerbaijan
* Sensitive to noisy or sparse features

#### **Gradient Boosting**

* Achieves **consistently lower MAE** across most events
* Demonstrates strong ability to model long-run tyre degradation
* Better captures the sequential nature of stint performance
* Handles outlier races more effectively than Random Forest

Overall, **Gradient Boosting outperforms Random Forest** in almost every race, confirming it is the stronger non-linear model for tyre-degradation prediction.

These results reinforce the conclusion that lap-time evolution is **non-linear**, and boosting-based models capture its complexity more effectively than both linear baselines and tree ensembles.


In [12]:
comparison_df = pd.DataFrame({
    'event': events,
    'Random Forest MAE': rf_event_df.sort_values('event')['MAE'].values,
    'Gradient Boosting MAE': gb_event_df.sort_values('event')['MAE'].values
})

comparison_df


,event,Random Forest MAE,Gradient Boosting MAE
0,Abu Dhabi Grand Prix,5.952772,3.337471
1,Australian Grand Prix,11.840630,9.867533
2,Austrian Grand Prix,20.345542,19.760305
3,Azerbaijan Grand Prix,14.673335,18.428641
4,Bahrain Grand Prix,4.934679,6.831006
5,Belgian Grand Prix,13.969267,18.120063
6,British Grand Prix,5.238039,5.451505
7,Canadian Grand Prix,10.373117,8.620021
8,Chinese Grand Prix,12.896982,16.274728
9,Dutch Grand Prix,15.760251,15.059566




# Key Findings

### **■ Gradient Boosting is overall better**

Most races have lower MAE using Gradient Boosting → stronger generalization.

Examples:

* **Abu Dhabi:** RF 5.95 → GB 3.33 (GB much better)
* **Canada:** RF 10.37 → GB 8.62 (GB better)
* **Hungary:** RF 10.17 → GB 7.25 (GB better)

### ■ But Random Forest wins on some tracks

Examples:

* **Azerbaijan:** RF 14.67 vs GB 18.42
* **Belgium:** RF 13.96 vs GB 18.12

These tracks have:

* long straights
* high-speed variability
* strong dependence on race-specific behaviour

Tree ensembles like RF sometimes handle noisy or weird tracks better.

### ■ Both models struggle on certain races

Very high MAE races:

* **Austria (~20 MAE)**
* **Azerbaijan (~18 MAE)**
* **Belgium (~18 MAE)**
* **China (~16 MAE)**

These likely have:

* unusual tyre behaviour
* extreme degradation phases
* strategy shifts
* high-frequency pace changes

This is normal — tyre deg is track-dependent.

---





##  Leave-One-Event-Out Evaluation (LOEO)

In this section, we evaluate how well our baseline models generalize to **unseen races**.
For each Grand Prix:

1. We remove that event from the training set.
2. Train Random Forest and Gradient Boosting on the remaining races.
3. Test on the held-out race.
4. Compute MAE.

This simulates a realistic scenario:

> *“If we go into a new race weekend, can our model predict tyre degradation without historical data from this track?”*

---

##  Results Summary

* **Gradient Boosting generally outperforms Random Forest**, achieving lower MAE on most races.
* **Random Forest performs better on a few circuits**, especially those with more unpredictable pace evolution (e.g., Azerbaijan, Belgium).
* **Some tracks are inherently harder to predict**, showing high error for both models (Austria, China).
* **Consistency matters**: Gradient Boosting delivers more stable results across races.

---


##  Interpretation

* Tracks with smooth degradation patterns → **Gradient Boosting excels**
* Tracks with chaotic or mixed tyre strategies → **Random Forest sometimes wins**
* High-error races indicate where:
  * Tyre deg is nonlinear
  * Degradation changes abruptly
  * Drivers manage tyres differently
  * Track temperature varies heavily


